# Score-Guided Manifold Projection (SGMP)
### Bridging Diffusion Priors and Discriminative Likelihood for Robust Pattern Recognition
**Target Venue:** *Pattern Recognition (Elsevier)* - Special Issue on Generative Models for Computer Vision (`VSI: PR_GCV`)

This notebook provides a complete, turnkey pipeline optimized for an **NVIDIA A100 GPU (40GB / 80GB SXM4/PCIe)** on Google Colab to train, benchmark, ablate, and export publication results for the SGMP framework.

---
### Key Technical Features Optimized for A100:
- **Hardware Tensor Cores:** PyTorch Automatic Mixed Precision (`bfloat16`) and TensorFloat-32 (`TF32`).
- **Full Dataset Scale:** 60,000 training patterns, 10,000 test patterns with 4 prefetch workers and pinned memory.
- **Exponential Moving Average (EMA):** Parameter smoothing on continuous-time ScoreNet for stable vector fields.
- **12-Family Corruption Suite:** Standardized Hendrycks benchmark (noise, blur, weather, occlusion) + white-box PGD attack.
- **Instant Paper Figures & Tables:** Auto-generates high-res vector PDFs/PNGs and LaTeX tables directly into `Paper/figs/`.

## 1. Hardware Verification & GPU Diagnostics

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version:    {torch.__version__}")
print(f"CUDA Available:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Device:             {gpu_name}")
    print(f"Total VRAM:         {vram:.2f} GB")
    print(f"bfloat16 Supported: {torch.cuda.is_bf16_supported()}")
    if "A100" in gpu_name:
        print("\u2705 Detected NVIDIA A100 Tensor Core GPU! Optimal configuration unlocked.")
    else:
        print(f"\u2139 Running on {gpu_name}. AMP will dynamically adapt.")

## 2. Environment Setup & Dependencies

In [ ]:
# If running directly from a cloned repo in Colab, navigate to Implementation
import os
if os.path.exists("Implementation"):
    os.chdir("Implementation")
print(f"Current Working Directory: {os.getcwd()}")

!pip install -q -r requirements.txt
print("\u2705 All dependencies satisfied.")

## 3. High-Throughput A100 Training Pipeline
Trains the full suite on 60,000 samples with `bfloat16` AMP, TensorFloat-32, CosineAnnealingLR, and ScoreNet EMA (~6–9 mins on A100).

In [ ]:
from config import Config
from train import run_training

# Apply the A100 high-performance profile
Config.apply_profile("a100")
print(f"Configured Device:    {Config.DEVICE}")
print(f"Batch Size:           {Config.BATCH_SIZE}")
print(f"Mixed Precision (AMP): {Config.USE_AMP} ({Config.AMP_DTYPE})")
print(f"Full 60k Dataset:     {Config.FULL_DATASET}")
print(f"ScoreNet EMA:         {Config.USE_EMA} (decay={Config.EMA_DECAY})")

# Launch training
run_training(profile="a100")

## 4. Comprehensive Out-of-Distribution & Adversarial Benchmark
Evaluates all 5 paradigms (Vanilla ERM, PGD-AT, DAE, DiffPure, SGMP) across 10,000 test samples under Clean, 11 Standardized Hendrycks Corruptions (Severity 3), and white-box Adversarial PGD attacks.

In [ ]:
from evaluate import BenchmarkEvaluator

evaluator = BenchmarkEvaluator(device=Config.DEVICE, batch_size=Config.BATCH_SIZE)
# Run full benchmark across all 10,000 test samples
benchmark_results = evaluator.run_full_benchmark(max_batches=None)

## 5. Ablation Studies (Step Budgets, Guidance Scale, Component Isolation)

In [ ]:
# Execute component breakdown, trajectory step budget, and guidance scale ablations
ablation_results = evaluator.run_ablation_study(max_batches=8)

## 6. Publication Figures Generation
Generates vector PDFs and 300-DPI PNGs for Elsevier Pattern Recognition into `../Paper/figs/`.

In [ ]:
from visualize import (
    generate_fig1_framework,
    generate_fig2_phase_plane,
    generate_fig3_robustness_radar,
    generate_fig4_qualitative_gallery,
    generate_fig5_ablations
)
from IPython.display import Image, display

print("Exporting figures...")
generate_fig1_framework()
generate_fig2_phase_plane()
generate_fig3_robustness_radar(benchmark_results)
generate_fig4_qualitative_gallery()
generate_fig5_ablations(ablation_results)

# Inline display of generated publication figures
figs = ["fig1_framework.png", "fig2_phase_plane.png", "fig3_robustness_radar.png", "fig4_qualitative_gallery.png", "fig5_ablations.png"]
for f in figs:
    p = os.path.join(Config.FIGS_DIR, f)
    if os.path.exists(p):
        print(f"\n--- {f} ---")
        display(Image(filename=p, width=750))

## 7. Formatted LaTeX Tables for Paper Submission (`main.tex`)

In [ ]:
print("=" * 85)
print("COPY-PASTE READY LATEX TABLE (Table 1: Main Benchmark Results)")
print("=" * 85)
print(r"\begin{tabular*}{\textwidth}{@{\extracolsep{\fill}}lcccccc@{}}")
print(r"\toprule")
print(r"\textbf{Method} & \textbf{Clean Acc (\%)} & \textbf{Corrupt Acc (\%)} & \textbf{mCE ($\downarrow$)} & \textbf{RRS ($\uparrow$)} & \textbf{PSNR (dB)} & \textbf{Latency (ms)} \\")
print(r"\midrule")
for m, d in benchmark_results.items():
    c_acc = f"{d.get('clean', 0):.2f}"
    cr_acc = f"{d.get('mean_corrupted_acc', 0):.2f}"
    mce = f"{d.get('mCE', 100):.1f}"
    rrs = f"{d.get('relative_robustness', 0):.3f}"
    psnr = f"{d.get('psnr', 0):.2f}" if 'psnr' in d else "--"
    lat = f"{d.get('avg_latency_ms', 0):.2f}"
    if m == "SGMP (Ours)":
        print(f"\\textbf{{{m}}} & \\textbf{{{c_acc}}} & \\textbf{{{cr_acc}}} & \\textbf{{{mce}}} & \\textbf{{{rrs}}} & \\textbf{{{psnr}}} & {lat} \\\\")
    else:
        print(f"{m} & {c_acc} & {cr_acc} & {mce} & {rrs} & {psnr} & {lat} \\\\")
print(r"\bottomrule")
print(r"\end{tabular*}")
print("=" * 85)

## 8. Download / Archive Checkpoints & Results
Zips the trained checkpoints, benchmark tables, and figures into a single downloadable archive.

In [ ]:
!zip -r sgmp_a100_results.zip checkpoints/ results/ ../Paper/figs/
print("\u2705 Archive 'sgmp_a100_results.zip' ready for download.")

# Optional: Trigger browser download if running in Google Colab
try:
    from google.colab import files
    files.download('sgmp_a100_results.zip')
except Exception:
    print("To download, use the Colab file explorer on the left pane and download 'sgmp_a100_results.zip'.")